## Notebook 8: Dynamic CBA

In [1]:
import os
os.environ["CITY"] = "rome"   # pick the city here

In [2]:
# Generic bootstrap 
from pathlib import Path
import os, sys

def _find_root():
    start = Path.cwd()
    for cand in [start, *start.parents]:
        if (cand/"cityheat").is_dir() and (cand/"configs").is_dir():
            return cand
    raise RuntimeError("Repo root not found.")
ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityheat.nbsetup import bootstrap
from cityheat.paths import make_P, ensure_out

# Choose city here
SLUG = globals().get("SLUG", os.environ.get("CITY", "rome")).lower()

C    = bootstrap(SLUG)      # reads configs/<slug>.yml and syncs that city only if wanted
CFG  = C["CFG"]; CITY = C["CITY"]
BASE = C["BASE"]; OUT = C["OUT"]; INT = C["INT"]

P    = make_P(BASE)         # read-only path helper
OUTP = ensure_out(OUT)      # write-safe path helper
print(f"→ City: {CITY}  |  BASE={BASE}  OUT={OUT}  INT={INT}")

→ City: rome  |  BASE=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome  OUT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome  INT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/interim


In [3]:
# city config + file paths from YAML 
cfg = C.get("cfg", {})                      # full YAML for the selected city
SLUG = cfg.get("slug", SLUG).lower()        
CITY = cfg.get("city_name", CITY)

paths    = cfg.get("files", {})             # {gvi_csv, lcz_candidates, cooling_coeffs_csv, ...}
osm_cfg  = cfg.get("osm", {})               # OSM settings used later in NB5
trees_cfg = cfg.get("trees", {})            # TARGET/CAP for NB5
urbclim   = cfg.get("urbclim", {})          # UrbClim folder/settings for NB5

lcz_candidates = [P(p) for p in paths.get("lcz_candidates", [])]
gvi_path = P(paths.get("gvi_csv", ""))

# FUA geopackage written in NB2 
fua_gpkg = Path(paths.get("fua_gpkg", f"{OUT}/{SLUG}_fua.gpkg"))

cool_csv = P(paths.get("cooling_coeffs_csv", ""))

# checks
print("SLUG/CITY:", SLUG, CITY)
print("GVI CSV:  ", gvi_path)
print("LCZ cand: ", [str(p) for p in lcz_candidates])
print("FUA GPKG: ", fua_gpkg)
print("Cooling CSV:", cool_csv)

SLUG/CITY: rome Rome
GVI CSV:   /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/gviRome/gvi_Rome.csv
LCZ cand:  ['/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_filter_v3.tif', '/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_v3.tif']
FUA GPKG:  /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/rome_fua.gpkg
Cooling CSV: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/CoolingEff/outer_2_wbgt_max.csv


**Loading from before**

- Trees policy table and citywide ΔGVI diagnostics (sum of Municipi)
- AC coverage maps + population grid
- Municipality-level coverage and kWh/user outputs from earlier notebooks
- AC efficacy by age (used on benefit side)

In [4]:
# loading everything needed for the CBA
from pathlib import Path
import json
import numpy as np
import pandas as pd

OUT = Path(OUT)
INT = Path(INT)
TAB_DIR = OUT / "tables"

# Vegetation policy / ΔGVI
trees_tbl = pd.read_csv(TAB_DIR / f"{SLUG}_trees_tbl.csv")

# diagnostics JSON with citywide ΔGVI (pop-weighted, points on 1–100 scale)
veg_diag_path = OUT / f"{SLUG}_veg_diagnostics.json"
veg_diag = json.loads(veg_diag_path.read_text())
citywide_dGVI_points_popw = veg_diag["citywide_dGVI_points_popw"]
print("Citywide pop-weighted ΔGVI (points, 1–100 scale):", citywide_dGVI_points_popw)

# AC coverage / municipal pop / energy use
# coverage maps
ac_cov_npz = np.load(INT / f"ac_coverage_maps_{SLUG}.npz")
coverage_base = ac_cov_npz["coverage_base"]
coverage_policy = ac_cov_npz["coverage_policy"]
CITY_MASK = ac_cov_npz["CITY_MASK"].astype(bool)
HGT = int(ac_cov_npz["HGT"])
WDT = int(ac_cov_npz["WDT"])

# population on ref grid
pop_npz = np.load(INT / f"pop_on_ref_{SLUG}.npz")
pop_on_ref = pop_npz["pop"]

# municipio coverage table (pop_muni, ac_base_muni, ac_policy_muni)
muni_cov = pd.read_csv(OUT / f"muni_cov_{SLUG}.csv")

# AC consumption per municipality (kWh per AC user)
muni_tbl_all = pd.read_csv(OUT / f"{SLUG}_muni_ac_consumption_summary.csv")

# AC efficacy by age (for benefits, not directly cost-side)
eff_buckets_path = INT / f"ac_eff_buckets_{SLUG}.json"
EFF_BUCKETS = json.loads(eff_buckets_path.read_text())
EFF_BUCKETS

Citywide pop-weighted ΔGVI (points, 1–100 scale): 2.3197


{'<15': 0.2, '15-64': 0.3, '65+': 0.4}

**Discount helpers**

- Set discount rate R and horizon T
- Define PV helpers and annuity factor (PV -> equivalent annual cost)
- Convention: flows are treated as end-of-year (years 1..T)

In [5]:
import numpy as np

R = 0.03 # discount rate
T = 25 # time horizon

TREE_RAMP_YEARS = 12          # maturity ramp for both benefits and O&M
TREE_START_AGE_CENTRAL = 5    # central case (pre-grown trees)
TREE_START_AGE_SENS = 0       # sensitivity 

TREE_START_AGE_YEARS = TREE_START_AGE_CENTRAL  # the one used by the model by default

# Convention: all flows occur at END of each year => years 1..T

# present value (PV) of a constant annual flow over T years.
def pv_level_flow(annual, r=R, T=T):
    """PV of a constant annual amount paid in years 1..T."""
    yrs = np.arange(1, T+1, dtype=float)
    return float(np.sum(annual * (1 + r) ** (-yrs)))

# PV of buying an item at t=0 and then replacing it every 'life' years within the horizon.
def pv_replacements(n_items, capex_per_item, r=R, T=T, life=20):
    """ PV of buying 'n_items' at t=0 and replacing every 'life' years within horizon T. """
    pv = 0.0
    t = 0
    while t <= T:
        pv += n_items * capex_per_item / ((1+r)**t if t > 0 else 1.0)
        t += life
    return float(pv)

# annuity_factor: present value of "1 euro per year" over T years. We later use this
# to convert any PV into a constant equivalent annual cost (EAC).
def annuity_factor(r=R, T=T):
    return (1 - (1 + r) ** (-T)) / r

AF = annuity_factor(R, T)
AF

17.413147691278027

In [6]:
def pv_capex_with_ramp(new_users_t, capex_per_user, life, r=R):
    """ PV of AC capex when policy coverage ramps up over time.

    new_users_t: 1D array length T, number of new policy users in each year (vs baseline).
    capex_per_user: installation cost per AC user.
    life: years between replacements.
    r: discount rate.

    For each cohort of new users in year t, we:
    - pay capex once at installation (year t+1 in our convention)
    - then pay the same capex again every life years (replacement)
    - discount each of these payments back to year 0 and sum them up.
    """
    T = len(new_users_t)
    pv = 0.0
    for t in range(T):  # t = 0..T-1 corresponds to years 1..T
        cohort = float(new_users_t[t])
        if cohort <= 0:
            continue
        pay_year = t  # installation in year t+1, then every 'life' years
        while pay_year < T:
            pv += cohort * capex_per_user / ((1 + r) ** (pay_year + 1))
            pay_year += life
    return float(pv)

**Trees: parametrisation of costs**

- Convert ΔGVI to “index points” and apply €/index-point CAPEX
- CAPEX: linear rollout over T years
- O&M: starts the year after planting and scales with maturity (ramp years)
- start_age shifts maturity (central=5, sensitivity=0)

- Compare alternative conventions:
    - A) O&M starts immediately (old)
    - B) O&M starts next year
    - C) O&M starts next year + maturity-scaled (current best practice)

In [7]:
# Parameters from rule and regreen study
CAPEX_PER_INDEX_PT = 10_000_000.0 # eur per 1 index point (1–100 scale)

# converting per-tree CAPEX and per-tree O&M into an O&M cost per GVI point.
CAPEX_PER_TREE = 210.0 # eur per tree (REGREEN median)
OM_PER_TREE_YR = 27.0 # eur per tree per year
LIFETIME_YEARS = 25 # tree benefit/O&M lifetime

# O&M per index point per year implied by tree-level numbers
OM_PER_INDEX_PT_YR = (OM_PER_TREE_YR / CAPEX_PER_TREE) * CAPEX_PER_INDEX_PT
print("O&M per index point per year (EUR):", round(OM_PER_INDEX_PT_YR, 0))

# Total change in GVI in index point (1–100 SCALE) from trees_tbl
# trees_tbl['dGVI'] is in 0–1 (fraction of max index); sum over Municipi, then ×100 => points
DELTA_INDEX = float(trees_tbl["dGVI"].clip(lower=0).sum()) * 100.0 # sum of municipio dGVI (0–1) => index points (0–100)
print("Total ΔGVI index points (1–100 scale):", round(DELTA_INDEX, 2))

# For comparison: citywide pop-weighted ΔGVI (diagnostics)
print("Pop-weighted ΔGVI points (diagnostic):", citywide_dGVI_points_popw)

# CAPEX: linear ramp over T years
# linear rollout over 25 years
# each year we add (ΔGVI / 25) points
# pay CAPEX once for that increment in that year
# investment schedule(t) = 10M€ * (ΔGVI/25) every year, discounted year by year.
def npv_capex_linear(delta_index_total, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT):
    """ PV of a linear ramp: we add delta_index_total/years index points each year over 'years',
    and pay capex_per_index per point. All flows are assumed at the end of years 1..years.
    """
    inc = delta_index_total / years  # index points added per year
    pv = 0.0
    for t in range(1, years + 1):  # t = 1..years
        capex_t = capex_per_index * inc
        pv += capex_t / ((1 + r) ** t)
    return float(pv)

# Paper-style "investment requirement" if all done at once (undiscounted)
TREES_CAPEX_T0 = CAPEX_PER_INDEX_PT * DELTA_INDEX

PV_trees_capex = npv_capex_linear(DELTA_INDEX, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT)
print(f"Trees — Total CAPEX requirement (undiscounted): €{TREES_CAPEX_T0:,.0f}")
print(f"Trees — NPV CAPEX (linear ramp): €{PV_trees_capex:,.0f}")

# annuity factor 
AF = annuity_factor(R, T)

# O&M that starts the year AFTER planting, and scales up with maturity (like benefits)
def cohort_rollout_maturity_factor_om(T, ramp_years, plant_share=None, lifetime=25, start_age_years=0):
    """
    factor[t] = sum_i plant_share[i] * maturity(age=t-i), where
    - planting year (age=0) has maturity forced to 0  => O&M starts at t+1
    - from age>=1, maturity is shifted by start_age_years (pre-grown trees)
    """
    if plant_share is None:
        plant_share = np.ones(T, dtype=float) / T

    max_age = min(T, lifetime)
    ages = np.arange(max_age + 1, dtype=float)  # 0..max_age

    maturity = np.minimum((ages + start_age_years) / ramp_years, 1.0)
    maturity[0] = 0.0  # <-- enforce: no O&M in planting year

    return np.convolve(plant_share, maturity)[:T]

def npv_om_cohorts_scaled(delta_index_total, years=T, r=R,
                          om_per_index_per_year=OM_PER_INDEX_PT_YR,
                          ramp_years=TREE_RAMP_YEARS,
                          lifetime=LIFETIME_YEARS,
                          start_age_years=0):
    plant_share = np.ones(years, dtype=float) / years

    factor_om = cohort_rollout_maturity_factor_om(
        T=years,
        ramp_years=ramp_years,
        plant_share=plant_share,
        lifetime=lifetime,
        start_age_years=start_age_years
    )

    om_stream = om_per_index_per_year * delta_index_total * factor_om
    yrs = np.arange(1, years + 1, dtype=float)
    pv = float(np.sum(om_stream * (1 + r) ** (-yrs)))
    return pv, om_stream

# call with the new shift
PV_trees_om, om_stream_scaled = npv_om_cohorts_scaled(
    DELTA_INDEX,
    years=T,
    r=R,
    om_per_index_per_year=OM_PER_INDEX_PT_YR,
    ramp_years=TREE_RAMP_YEARS,
    lifetime=LIFETIME_YEARS,
    start_age_years=TREE_START_AGE_YEARS,   
)

PV_trees_total = PV_trees_capex + PV_trees_om

EAC_capex_annuity = PV_trees_capex / AF
EAC_om_annuity    = PV_trees_om    / AF
EAC_total_annuity = PV_trees_total / AF

TREES_CAPEX_T0 = CAPEX_PER_INDEX_PT * DELTA_INDEX
EAC_capex_paper = TREES_CAPEX_T0 / ((1 + R)**T * T)

print(f"Trees — NPV O&M (scaled with maturity):       €{PV_trees_om:,.0f}")
print(f"Trees — NPV total (CAPEX + O&M):             €{PV_trees_total:,.0f}")
print(f"Trees — EAC CAPEX (annuity):                 €{EAC_capex_annuity:,.0f}/yr")
print(f"Trees — EAC O&M (annuity):                   €{EAC_om_annuity:,.0f}/yr")
print(f"Trees — EAC total (annuity):                 €{EAC_total_annuity:,.0f}/yr")

O&M per index point per year (EUR): 1285714.0
Total ΔGVI index points (1–100 scale): 30.23
Pop-weighted ΔGVI points (diagnostic): 2.3197
Trees — Total CAPEX requirement (undiscounted): €302,333,765
Trees — NPV CAPEX (linear ramp): €210,583,300
Trees — NPV O&M (scaled with maturity):       €243,071,452
Trees — NPV total (CAPEX + O&M):             €453,654,751
Trees — EAC CAPEX (annuity):                 €12,093,351/yr
Trees — EAC O&M (annuity):                   €13,959,076/yr
Trees — EAC total (annuity):                 €26,052,427/yr


In [8]:
print("om_stream first 5:", np.round(om_stream_scaled[:5], 2))
print("om_stream last  5:", np.round(om_stream_scaled[-5:], 2))
print("om_stream max:", float(np.max(om_stream_scaled)))
print("om_stream last:", float(om_stream_scaled[-1]))

om_stream first 5: [      0.    777429.68 1684430.97 2721003.88 3887148.4 ]
om_stream last  5: [28376183.34 29931042.7  31485902.06 33040761.42 34595620.78]
om_stream max: 34595620.77839674
om_stream last: 34595620.77839674


In [9]:
import numpy as np
import pandas as pd

def pv_from_stream(stream, r, T):
    yrs = np.arange(1, T+1, dtype=float)
    return float(np.sum(np.asarray(stream, float) * (1 + r) ** (-yrs)))

def om_stream_constant(delta_index_total, om_per_index_per_year, T, include_planting_year: bool):
    # Constant O&M per cohort (no maturity scaling)
    inc = delta_index_total / T
    stream = np.zeros(T, dtype=float)
    for t in range(1, T+1):  # cashflows in years 1..T
        active = (t if include_planting_year else max(t-1, 0))
        stream[t-1] = active * om_per_index_per_year * inc
    return stream

def om_stream_scaled(delta_index_total, om_per_index_per_year, T, ramp_years):
    # Scaled with “maturity”, and zero in planting year by construction
    plant_share = np.ones(T, dtype=float) / T
    ages = np.arange(T+1, dtype=float)              # 0..T
    maturity = np.minimum(ages / ramp_years, 1.0)   # age 0 => 0
    factor = np.convolve(plant_share, maturity)[:T]
    return om_per_index_per_year * delta_index_total * factor

TREE_OM_RAMP_YEARS = TREE_RAMP_YEARS

# A) very old convention: O&M starts immediately
sA = om_stream_constant(DELTA_INDEX, OM_PER_INDEX_PT_YR, T, include_planting_year=True)
PV_A = pv_from_stream(sA, R, T)

# B) point 1: O&M starts the year after planting
sB = om_stream_constant(DELTA_INDEX, OM_PER_INDEX_PT_YR, T, include_planting_year=False)
PV_B = pv_from_stream(sB, R, T)

# C) point 2: O&M starts next year AND scales with maturity
sC = om_stream_scaled(DELTA_INDEX, OM_PER_INDEX_PT_YR, T, ramp_years=TREE_OM_RAMP_YEARS)
PV_C = pv_from_stream(sC, R, T)

rows = []
for name, PV_om, stream in [
    ("A) O&M starts same year (old)", PV_A, sA),
    ("B) O&M starts next year", PV_B, sB),
    ("C) O&M next year + maturity-scaled", PV_C, sC),
]:
    PV_total = PV_trees_capex + PV_om
    rows.append({
        "scenario": name,
        "PV_capex": PV_trees_capex,
        "PV_om": PV_om,
        "PV_total": PV_total,
        "EAC_capex": PV_trees_capex / AF,
        "EAC_om": PV_om / AF,
        "EAC_total": PV_total / AF,
        "PV_om_over_PV_capex": PV_om / PV_trees_capex,
        "OM_year1": stream[0],
        "OM_yearT": stream[-1],
    })

compare_om = pd.DataFrame(rows)
compare_om

,scenario,PV_capex,PV_om,PV_total,EAC_capex,EAC_om,EAC_total,PV_om_over_PV_capex,OM_year1,OM_yearT
0,A) O&M starts same year (old),2.105833e+08,3.107336e+08,5.213169e+08,1.209335e+07,1.784477e+07,2.993812e+07,1.475585,1.554859e+06,3.887148e+07
1,B) O&M starts next year,2.105833e+08,2.836586e+08,4.942419e+08,1.209335e+07,1.628991e+07,2.838326e+07,1.347014,0.000000e+00,3.731662e+07
2,C) O&M next year + maturity-scaled,2.105833e+08,1.682367e+08,3.788200e+08,1.209335e+07,9.661474e+06,2.175482e+07,0.798908,0.000000e+00,2.876490e+07


- Trees O&M sensitivity: force PV(O&M) = 5 × PV(CAPEX)
   - Compute multiplier on O&M rate so the PV ratio hits the target
   - Used for “high O&M” sensitivity cases in summary tables

In [10]:
# point 3: sensitivity where PV(O&M) = 5 * PV(CAPEX) under the SAME timing convention 

TARGET_OM_TO_CAPEX_PV_RATIO = 5.0   # sensitivity

# baseline under scenario C (O&M starts next year + maturity scaling)
PV_om_base, om_stream_base = npv_om_cohorts_scaled(
    DELTA_INDEX,
    years=T,
    r=R,
    om_per_index_per_year=OM_PER_INDEX_PT_YR,
    ramp_years=TREE_OM_RAMP_YEARS,
    lifetime=LIFETIME_YEARS
)

# scale factor to hit target PV ratio
k = (TARGET_OM_TO_CAPEX_PV_RATIO * PV_trees_capex) / PV_om_base
OM_PER_INDEX_PT_YR_5x = OM_PER_INDEX_PT_YR * k

PV_om_5x, om_stream_5x = npv_om_cohorts_scaled(
    DELTA_INDEX,
    years=T,
    r=R,
    om_per_index_per_year=OM_PER_INDEX_PT_YR_5x,
    ramp_years=TREE_OM_RAMP_YEARS,
    lifetime=LIFETIME_YEARS
)

print("O&M sensitivity scaling (scenario C timing):")
print(f"  PV(CAPEX)                 = €{PV_trees_capex:,.0f}")
print(f"  Base PV(O&M)              = €{PV_om_base:,.0f}")
print(f"  Target PV(O&M)            = €{TARGET_OM_TO_CAPEX_PV_RATIO * PV_trees_capex:,.0f}")
print(f"  Multiplier k on OM rate   = {k:.2f}x")
print(f"  New PV(O&M)               = €{PV_om_5x:,.0f}")

# optional: EAC comparison
print(f"  Base EAC(O&M)             = €{PV_om_base/AF:,.0f}/yr")
print(f"  New  EAC(O&M)             = €{PV_om_5x/AF:,.0f}/yr")

O&M sensitivity scaling (scenario C timing):
  PV(CAPEX)                 = €210,583,300
  Base PV(O&M)              = €168,236,673
  Target PV(O&M)            = €1,052,916,499
  Multiplier k on OM rate   = 6.26x
  New PV(O&M)               = €1,052,916,499
  Base EAC(O&M)             = €9,661,474/yr
  New  EAC(O&M)             = €60,466,753/yr


**Cost AC**

- AC cost model (dynamic coverage + dynamic kWh/user)
  - Coverage ramps by municipality over time (policy - baseline)
  - New users cohorts drive CAPEX and replacement cycles
  - Ongoing costs: maintenance + electricity (kWh/user interpolated over horizon)

In [11]:
# Dynamic AC horizon years (aligned with NB7 / CLIMADA outputs)
# We only need the start year and the list of years for the CBA horizon.
ac_city_series = pd.read_csv(INT / f"ac_per_user_city_{SLUG}.csv")
ac_city_series = ac_city_series.set_index("year").sort_index()
ELEC_START_YEAR = int(ac_city_series.index.min()) # should be 2030
ELEC_YEARS = np.arange(ELEC_START_YEAR, ELEC_START_YEAR + T, dtype=int)

In [12]:
# AC COSTS: dynamic coverage + dynamic kWh/user

# AC PARAMS
AC_CAPEX_PER_USER = 500.0 # € per AC unit
AC_MAINT_RATE = 0.05 # fraction of CAPEX per year
AC_LIFETIME_YEARS = 10 # replacement cycle
TARIFF_EUR_PER_KWH = 0.25 # €/kWh

# per-user annual maintenance
maint_per_user_yr = AC_MAINT_RATE * AC_CAPEX_PER_USER

# Horizon years (should match benefits horizon: 2030..2030+T-1)
YEARS = ELEC_YEARS.copy()
assert len(YEARS) == T

# Municipio level coverage over time
# Table with coverage by year and municipality (cf notebook 5, we have it there)
try:
    muni_cov_all = pd.read_csv(OUT / f"{SLUG}_muni_cov_yearly.csv")
except FileNotFoundError:
    muni_cov_all = muni_cov.copy()

# we keep only rows for which coverage is defined
# and, we can, restrict to Municipi inside the city proper
if "muni_id" in muni_cov_all.columns:
    muni_cov_all = muni_cov_all.loc[muni_cov_all["muni_id"] > 0].copy()

# additional AC share per municipio and year (policy vs baseline)
muni_cov_all["dshare"] = (
    muni_cov_all["ac_policy_muni"] - muni_cov_all["ac_base_muni"]
).clip(lower=0.0)

# interpolating dshare to all years for each municipality
rows = []
for muni_id, g in muni_cov_all.groupby("muni_id"):
    g = g.sort_values("year")
    known_years = g["year"].to_numpy(int)
    known_dshare = g["dshare"].to_numpy(float)
    pop_muni = float(g["pop_muni"].iloc[0]) # assume pop_muni constant over time

    dshare_t = np.interp(YEARS, known_years, known_dshare) # flat before first and after last known year
    dshare_t[YEARS <= known_years[0]] = known_dshare[0]
    dshare_t[YEARS >= known_years[-1]] = known_dshare[-1]

    for year, ds in zip(YEARS, dshare_t):
        rows.append({
            "year": year,
            "muni_id": muni_id,
            "pop_muni": pop_muni,
            "dshare_t": ds,
        })

# coverage ramp
cov_yearly = pd.DataFrame(rows)

# total policy AC users per year (vs baseline)
added_users_t = (
    cov_yearly
    .assign(users=lambda d: d["pop_muni"] * d["dshare_t"])
    .groupby("year")["users"]
    .sum()
    .reindex(YEARS)
    .to_numpy(float)
)

# added_users_t: total incremental AC users each year vs baseline (from coverage ramp)
# new_users_t: year-to-year increments (cohorts) -> used for CAPEX with replacements
new_users_t = np.empty_like(added_users_t)
new_users_t[0] = added_users_t[0]
new_users_t[1:] = np.maximum(added_users_t[1:] - added_users_t[:-1], 0.0)
added_users_final = float(added_users_t[-1])
print(f"AC — added users in final year ≈ {added_users_final:,.0f}")

# Electricity costs: dynamic coverage + dynamic kWh/user from NB7

# muni-level kWh per AC user from NB7 (available for 2030, 2040, 2050)
muni_tbl_all = pd.read_csv(OUT / f"{SLUG}_muni_ac_consumption_summary.csv")
years_full = YEARS # array([2030, ..., 2054])

rows_kwh = []
for muni_id, g in muni_tbl_all.groupby("muni_id"):
    g = g.sort_values("year")
    known_years = g["year"].to_numpy(int)
    vals = g["kwh_per_user_muni"].to_numpy(float)

    # interpolate to the full horizon
    kwh_interp = np.interp(years_full, known_years, vals) # flat before first and after last known year
    kwh_interp[years_full <= known_years[0]] = vals[0]
    kwh_interp[years_full >= known_years[-1]] = vals[-1]

    for y, v in zip(years_full, kwh_interp):
        rows_kwh.append({
            "year": y,
            "muni_id": muni_id,
            "kwh_per_user_muni": v,
        })

muni_kwh_full = pd.DataFrame(rows_kwh)

# attach interpolated kWh/user to coverage ramp
cov_yearly = cov_yearly.merge(
    muni_kwh_full, on=["year", "muni_id"], how="left"
).fillna({"kwh_per_user_muni": 0.0})

# kWh per year: pop * extra AC share * kWh per AC user
cov_yearly["kwh_t"] = (
    cov_yearly["pop_muni"] * cov_yearly["dshare_t"] * cov_yearly["kwh_per_user_muni"]
)

elec_eur_t = (
    cov_yearly.groupby("year")["kwh_t"].sum()
    .reindex(YEARS)
    .to_numpy(float)
) * TARIFF_EUR_PER_KWH

# discounted PV of elec, capex, maintenance
yrs = np.arange(1, T+1, dtype=float) # 1..25

# Electricity is a yearly flow: dynamic users * dynamic kWh/user * tariff
PV_ac_elec = float(np.sum(elec_eur_t * (1 + R) ** (-yrs)))
print(f"AC — PV elec (dynamic coverage & kWh/user): €{PV_ac_elec:,.0f}")

# maintenance on all active policy users
maint_eur_t = added_users_t * maint_per_user_yr
PV_ac_maint = float(np.sum(maint_eur_t * (1 + R) ** (-yrs)))
print(f"AC — PV maint €{PV_ac_maint:,.0f}")

# capex: cohorts of new users, with replacements every AC_LIFETIME_YEARS
PV_ac_capex = pv_capex_with_ramp(
    new_users_t,
    capex_per_user=AC_CAPEX_PER_USER,
    life=AC_LIFETIME_YEARS,
    r=R,
)
print(f"AC — PV capex €{PV_ac_capex:,.0f}")

# Total PV and EAC
PV_ac_total = PV_ac_capex + PV_ac_maint + PV_ac_elec
print(f"AC — PV capex €{PV_ac_capex:,.0f}")
print(f"AC — PV maint €{PV_ac_maint:,.0f}")
print(f"AC — PV elec €{PV_ac_elec:,.0f}")
print(f"AC — PV total €{PV_ac_total:,.0f}")

EAC_ac_total = PV_ac_total / AF
print(f"AC — EAC total (annuity): €{EAC_ac_total:,.0f}/yr")

AC — added users in final year ≈ 199,063
AC — PV elec (dynamic coverage & kWh/user): €868,871,567
AC — PV maint €93,209,551
AC — PV capex €281,703,895
AC — PV capex €281,703,895
AC — PV maint €93,209,551
AC — PV elec €868,871,567
AC — PV total €1,243,785,014
AC — EAC total (annuity): €71,427,925/yr


In [13]:
print("First 5 years of elec_eur_t:", elec_eur_t[:5])
print("Last 5 years of elec_eur_t:", elec_eur_t[-5:])

First 5 years of elec_eur_t: [58227575.18681791 56972894.72012631 55718264.92995242 54463685.81629623
 53209157.37915776]
Last 5 years of elec_eur_t: [47960842.10582834 47960842.10582834 47960842.10582834 47960842.10582834
 47960842.10582834]


**Benefits and summary**

- Health benefits time series (avoided heat deaths)
  - Read annual avoided deaths outputs (trees, AC, both, and trees-on-top)
  - Interpolate to full horizon
  - Apply tree rollout + maturity factor (dynamic) vs “full from start” (static)

In [14]:
# Benefits and summary
import numpy as np
import pandas as pd
from pathlib import Path

HORIZON_YEARS = T
DISCOUNT_RATE = R

INT = Path(INT)
USE_SCALED_BENEFITS = False

def _nb6_path(stem: str) -> Path:
    return INT / (f"{stem}_scaled_{SLUG}.csv" if USE_SCALED_BENEFITS else f"{stem}_{SLUG}.csv")

def _read_overall(stem: str) -> pd.Series:
    p = _nb6_path(stem)
    if not p.exists():
        raise FileNotFoundError(f"Missing NB6 output: {p}")
    s = pd.read_csv(p, index_col="year")["overall"]
    s.index = s.index.astype(int)
    return s.sort_index()

def pv_of_stream(cashflows, r=DISCOUNT_RATE):
    yrs = np.arange(1, len(cashflows) + 1, dtype=float)
    return float(np.sum(np.asarray(cashflows, float) * (1 + r) ** (-yrs)))

def interpolate_to_horizon(s: pd.Series, years: np.ndarray) -> np.ndarray:
    s = s.sort_index()
    known = s.index.to_numpy(int)
    vals = s.to_numpy(float)
    out = np.interp(years, known, vals)
    out[years <= known[0]] = vals[0]
    out[years >= known[-1]] = vals[-1]
    return out

avo_trees = _read_overall("annual_heat_deaths_avoided_trees_curr_AC")
avo_ac = _read_overall("annual_heat_deaths_avoided_AC_curr_AC")
avo_both = _read_overall("annual_heat_deaths_avoided_treesplusAC_curr_AC")

try:
    avo_trees_on_top = _read_overall("annual_heat_deaths_avoided_trees_on_top_AC_curr_AC")
except FileNotFoundError:
    avo_trees_on_top = (avo_both - avo_ac).rename("overall")

# these are:
# If the full trees intervention (full ΔGVI map) were already in place
# (and effectively delivering its modeled cooling),
# what deaths would it avoid in each climate year?”
print("Trees – avoided deaths per year:")
print(avo_trees, "\n")
print("AC policy – avoided deaths per year:")
print(avo_ac, "\n")
print("Both (trees+AC vs current AC) – avoided deaths per year:")
print(avo_both, "\n")
print("Trees on top of AC (NB6 file) – avoided deaths per year:")
print(avo_trees_on_top)

# Horizon construction
START_YEAR = int(min(avo_trees.index.min(), avo_ac.index.min(), avo_both.index.min(), avo_trees_on_top.index.min()))
YEARS = np.arange(START_YEAR, START_YEAR + HORIZON_YEARS, dtype=int)

trees_full = interpolate_to_horizon(avo_trees, YEARS)
ac_full = interpolate_to_horizon(avo_ac, YEARS)
both_full = interpolate_to_horizon(avo_both, YEARS)
top_full = interpolate_to_horizon(avo_trees_on_top, YEARS)

# Trees benefits: cohort rollout (25y) + maturity (12y) + optional "pre-grown" shift
def cohort_rollout_maturity_factor(T, ramp_years, plant_share=None, start_age_years=0):
    """
    factor[t] = sum_i plant_share[i] * maturity(age=t-i), where:
      - maturity(age=0) is forced to 0  -> benefits start at t+1
      - maturity for age>=1 is shifted by start_age_years (pre-grown trees)
    """
    if plant_share is None:
        plant_share = np.ones(T, dtype=float) / T  # linear rollout over T years

    ages = np.arange(T + 1, dtype=float)  # 0..T (need +1 so age=0 exists explicitly)
    maturity = np.minimum((ages + start_age_years) / ramp_years, 1.0)
    maturity[0] = 0.0  # enforce: no benefits in planting year

    return np.convolve(plant_share, maturity)[:T]

trees_factor_dynamic = cohort_rollout_maturity_factor(
    T=HORIZON_YEARS,
    ramp_years=TREE_RAMP_YEARS,
    start_age_years=TREE_START_AGE_YEARS
)

trees_factor_static = np.ones(HORIZON_YEARS, dtype=float)

def compute_streams(trees_full, ac_full, both_full, top_full, trees_factor):
    trees = trees_full * trees_factor
    ac    = ac_full
    top   = top_full * trees_factor
    both  = ac + top
    return trees, ac, top, both
    
trees_dyn, ac_dyn, top_dyn, both_dyn = compute_streams(trees_full, ac_full, both_full, top_full, trees_factor_dynamic)
trees_sta, ac_sta, top_sta, both_sta = compute_streams(trees_full, ac_full, both_full, top_full, trees_factor_static)

def summarize_benefits(prefix, trees, ac, top, both, r=DISCOUNT_RATE):
    pv = {
        f"{prefix}_PV_trees": pv_of_stream(trees, r),
        f"{prefix}_PV_ac": pv_of_stream(ac, r),
        f"{prefix}_PV_top": pv_of_stream(top, r),
        f"{prefix}_PV_both": pv_of_stream(both, r),
    }
    cum = {
        f"{prefix}_CUM_trees": float(np.sum(trees)),
        f"{prefix}_CUM_ac": float(np.sum(ac)),
        f"{prefix}_CUM_top": float(np.sum(top)),
        f"{prefix}_CUM_both": float(np.sum(both)),
    }
    return pv, cum

pv_dyn, cum_dyn = summarize_benefits("DYN", trees_dyn, ac_dyn, top_dyn, both_dyn)
pv_sta, cum_sta = summarize_benefits("STA", trees_sta, ac_sta, top_sta, both_sta)

print("Cumulative avoided deaths (25y, undiscounted):")
print(
    f" DYNAMIC — Trees: {cum_dyn['DYN_CUM_trees']:.2f} | AC: {cum_dyn['DYN_CUM_ac']:.2f} | "
    f"Trees on top: {cum_dyn['DYN_CUM_top']:.2f} | Both: {cum_dyn['DYN_CUM_both']:.2f}"
)
print(
    f" STATIC — Trees: {cum_sta['STA_CUM_trees']:.2f} | AC: {cum_sta['STA_CUM_ac']:.2f} | "
    f"Trees on top: {cum_sta['STA_CUM_top']:.2f} | Both: {cum_sta['STA_CUM_both']:.2f}"
)
print("\nNote: 'STATIC' assumes full policy effect from the first year of the horizon.")
print(" 'DYNAMIC' assumes gradual rollout + tree maturity, so early benefits are smaller.")

Trees – avoided deaths per year:
year
2030    8.700125
2040    9.050604
2050    9.537256
Name: overall, dtype: float64 

AC policy – avoided deaths per year:
year
2030    32.185809
2040    28.150387
2050    30.050515
Name: overall, dtype: float64 

Both (trees+AC vs current AC) – avoided deaths per year:
year
2030    40.229810
2040    36.635876
2050    38.991879
Name: overall, dtype: float64 

Trees on top of AC (NB6 file) – avoided deaths per year:
year
2030    8.044001
2040    8.485489
2050    8.941364
Name: overall, dtype: float64
Cumulative avoided deaths (25y, undiscounted):
 DYNAMIC — Trees: 97.55 | AC: 744.01 | Trees on top: 91.41 | Both: 835.42
 STATIC — Trees: 228.96 | AC: 744.01 | Trees on top: 214.04 | Both: 958.05

Note: 'STATIC' assumes full policy effect from the first year of the horizon.
 'DYNAMIC' assumes gradual rollout + tree maturity, so early benefits are smaller.


- Baseline mortality and percentage reductions
- 
We now read the baseline heat-attributable deaths with current AC (no new policy) from CLIMADA and:
    - compute total baseline deaths per year (summing over age classes)
    - align avoided deaths for trees, AC, and both to these baseline years
    - compute percentage reductions in baseline deaths:
      * trees_pct = % reduction with trees only
      * ac_pct    = % reduction with AC only
      * both_pct  = % reduction with both policies
      * trees_on_top_pct = extra % reduction from adding trees on top of AC

In [15]:
# building a clean "tree cost pack" for base vs 5x O&M (same timing convention) 

def pack_tree_costs(PV_capex, PV_om, AF, TREES_CAPEX_T0, R, T):
    PV_total = PV_capex + PV_om
    return {
        "PV_capex": PV_capex,
        "PV_om": PV_om,
        "PV_total": PV_total,
        "EAC_capex_annuity": PV_capex / AF,
        "EAC_om_annuity": PV_om / AF,
        "EAC_total_annuity": PV_total / AF,
        # paper-style: capex-only shortcut, unchanged by O&M scenario
        "EAC_capex_paper": (TREES_CAPEX_T0 / ((1 + R) ** T)) / T,
    }

trees_cost_base = pack_tree_costs(PV_trees_capex, PV_trees_om, AF, TREES_CAPEX_T0, R, T)
trees_cost_5x   = pack_tree_costs(PV_trees_capex, PV_om_5x,    AF, TREES_CAPEX_T0, R, T)

In [16]:
# Summary table builder that takes a tree-cost scenario explicitly 

def safe_ratio(c, b):
    return float(c / b) if (b is not None and b > 1e-9) else np.inf

def build_summary(label, cost_scenario, trees_cost, CUM_trees, CUM_ac, CUM_both, CUM_top):
    return pd.DataFrame([
        {
            "Benefit_timing": label,
            "Cost_scenario": cost_scenario,
            "Policy": "Trees only (vs current AC)",
            "PV_cost_eur": trees_cost["PV_total"],
            "avoided_deaths_cum": CUM_trees,
            "Cost_per_avoided_death_eur": safe_ratio(trees_cost["PV_total"], CUM_trees),
            "EAC_capex_annuity_eur_per_yr": trees_cost["EAC_capex_annuity"],
            "EAC_om_annuity_eur_per_yr":    trees_cost["EAC_om_annuity"],
            "EAC_total_annuity_eur_per_yr": trees_cost["EAC_total_annuity"],
            "EAC_capex_paper_eur_per_yr":   trees_cost["EAC_capex_paper"],
            "added_AC_users": 0.0,
        },
        {
            "Benefit_timing": label,
            "Cost_scenario": cost_scenario,
            "Policy": "AC only (vs current AC)",
            "PV_cost_eur": PV_ac_total,
            "avoided_deaths_cum": CUM_ac,
            "Cost_per_avoided_death_eur": safe_ratio(PV_ac_total, CUM_ac),
            "EAC_capex_annuity_eur_per_yr": 0.0,
            "EAC_om_annuity_eur_per_yr":    0.0,
            "EAC_total_annuity_eur_per_yr": EAC_ac_total,
            "EAC_capex_paper_eur_per_yr":   np.nan,
            "added_AC_users": added_users_final,
        },
        {
            "Benefit_timing": label,
            "Cost_scenario": cost_scenario,
            "Policy": "Both (trees+AC vs current AC)",
            "PV_cost_eur": trees_cost["PV_total"] + PV_ac_total,
            "avoided_deaths_cum": CUM_both,
            "Cost_per_avoided_death_eur": safe_ratio(trees_cost["PV_total"] + PV_ac_total, CUM_both),
            "EAC_capex_annuity_eur_per_yr": trees_cost["EAC_capex_annuity"],
            "EAC_om_annuity_eur_per_yr":    trees_cost["EAC_om_annuity"],
            "EAC_total_annuity_eur_per_yr": trees_cost["EAC_total_annuity"] + EAC_ac_total,
            "EAC_capex_paper_eur_per_yr":   trees_cost["EAC_capex_paper"],
            "added_AC_users": added_users_final,
        },
        {
            "Benefit_timing": label,
            "Cost_scenario": cost_scenario,
            "Policy": "Trees (incremental, on top of AC policy)",
            "PV_cost_eur": trees_cost["PV_total"],
            "avoided_deaths_cum": CUM_top,
            "Cost_per_avoided_death_eur": safe_ratio(trees_cost["PV_total"], CUM_top),
            "EAC_capex_annuity_eur_per_yr": trees_cost["EAC_capex_annuity"],
            "EAC_om_annuity_eur_per_yr":    trees_cost["EAC_om_annuity"],
            "EAC_total_annuity_eur_per_yr": trees_cost["EAC_total_annuity"],
            "EAC_capex_paper_eur_per_yr":   trees_cost["EAC_capex_paper"],
            "added_AC_users": 0.0,
        },
    ])

# building base + sensitivity summaries for BOTH benefit-timing conventions 

summary_dynamic_base = build_summary(
    "Dynamic rollout + tree maturity",
    "Base O&M",
    trees_cost_base,
    cum_dyn["DYN_CUM_trees"], cum_dyn["DYN_CUM_ac"], cum_dyn["DYN_CUM_both"], cum_dyn["DYN_CUM_top"]
)

summary_dynamic_5x = build_summary(
    "Dynamic rollout + tree maturity",
    "O&M PV = 5× CAPEX PV",
    trees_cost_5x,
    cum_dyn["DYN_CUM_trees"], cum_dyn["DYN_CUM_ac"], cum_dyn["DYN_CUM_both"], cum_dyn["DYN_CUM_top"]
)

summary_static_base = build_summary(
    "Static (full effect from start)",
    "Base O&M",
    trees_cost_base,
    cum_sta["STA_CUM_trees"], cum_sta["STA_CUM_ac"], cum_sta["STA_CUM_both"], cum_sta["STA_CUM_top"]
)

summary_static_5x = build_summary(
    "Static (full effect from start)",
    "O&M PV = 5× CAPEX PV",
    trees_cost_5x,
    cum_sta["STA_CUM_trees"], cum_sta["STA_CUM_ac"], cum_sta["STA_CUM_both"], cum_sta["STA_CUM_top"]
)

summary_all = pd.concat(
    [summary_dynamic_base, summary_dynamic_5x, summary_static_base, summary_static_5x],
    ignore_index=True
).round(2)

summary_all

,Benefit_timing,Cost_scenario,Policy,PV_cost_eur,avoided_deaths_cum,Cost_per_avoided_death_eur,EAC_capex_annuity_eur_per_yr,EAC_om_annuity_eur_per_yr,EAC_total_annuity_eur_per_yr,EAC_capex_paper_eur_per_yr,added_AC_users
0,Dynamic rollout + tree maturity,Base O&M,Trees only (vs current AC),4.536548e+08,97.55,4650345.49,12093350.58,13959075.97,2.605243e+07,5775851.59,0.00
1,Dynamic rollout + tree maturity,Base O&M,AC only (vs current AC),1.243785e+09,744.01,1671741.22,0.00,0.00,7.142793e+07,NaN,199063.06
2,Dynamic rollout + tree maturity,Base O&M,Both (trees+AC vs current AC),1.697440e+09,835.42,2031851.24,12093350.58,13959075.97,9.748035e+07,5775851.59,199063.06
3,Dynamic rollout + tree maturity,Base O&M,"Trees (incremental, on top of AC policy)",4.536548e+08,91.41,4962875.05,12093350.58,13959075.97,2.605243e+07,5775851.59,0.00
4,Dynamic rollout + tree maturity,O&M PV = 5× CAPEX PV,Trees only (vs current AC),1.263500e+09,97.55,12951943.24,12093350.58,60466752.92,7.256010e+07,5775851.59,0.00
5,Dynamic rollout + tree maturity,O&M PV = 5× CAPEX PV,AC only (vs current AC),1.243785e+09,744.01,1671741.22,0.00,0.00,7.142793e+07,NaN,199063.06
6,Dynamic rollout + tree maturity,O&M PV = 5× CAPEX PV,Both (trees+AC vs current AC),2.507285e+09,835.42,3001243.31,12093350.58,60466752.92,1.439880e+08,5775851.59,199063.06
7,Dynamic rollout + tree maturity,O&M PV = 5× CAPEX PV,"Trees (incremental, on top of AC policy)",1.263500e+09,91.41,13822387.19,12093350.58,60466752.92,7.256010e+07,5775851.59,0.00
8,Static (full effect from start),Base O&M,Trees only (vs current AC),4.536548e+08,228.96,1981365.51,12093350.58,13959075.97,2.605243e+07,5775851.59,0.00
9,Static (full effect from start),Base O&M,AC only (vs current AC),1.243785e+09,744.01,1671741.22,0.00,0.00,7.142793e+07,NaN,199063.06


In [17]:
def run_tree_age_case(tree_start_age_years: int):
    # TREE COSTS (capex unchanged; O&M depends on start age)
    PV_trees_om_case, _ = npv_om_cohorts_scaled(
        DELTA_INDEX,
        years=T,
        r=R,
        om_per_index_per_year=OM_PER_INDEX_PT_YR,
        ramp_years=TREE_RAMP_YEARS,
        lifetime=LIFETIME_YEARS,
        start_age_years=tree_start_age_years,
    )
    trees_cost_case = pack_tree_costs(
        PV_trees_capex, PV_trees_om_case, AF, TREES_CAPEX_T0, R, T
    )

    # TREE BENEFITS (dynamic rollout + maturity, shifted by start age) 
    trees_factor_dynamic_case = cohort_rollout_maturity_factor(
        T=HORIZON_YEARS,
        ramp_years=TREE_RAMP_YEARS,
        start_age_years=tree_start_age_years,
    )
    trees_dyn_case, ac_dyn_case, top_dyn_case, both_dyn_case = compute_streams(
        trees_full, ac_full, both_full, top_full, trees_factor_dynamic_case
    )
    _, cum_case = summarize_benefits("DYN", trees_dyn_case, ac_dyn_case, top_dyn_case, both_dyn_case)

    return build_summary(
        f"Dynamic rollout + maturity (start_age={tree_start_age_years})",
        "Base O&M",
        trees_cost_case,
        cum_case["DYN_CUM_trees"], cum_case["DYN_CUM_ac"], cum_case["DYN_CUM_both"], cum_case["DYN_CUM_top"],
    )

summary_central = run_tree_age_case(5)
summary_sens0   = run_tree_age_case(0)

summary_tree_age = pd.concat([summary_central, summary_sens0], ignore_index=True).round(2)
summary_tree_age

,Benefit_timing,Cost_scenario,Policy,PV_cost_eur,avoided_deaths_cum,Cost_per_avoided_death_eur,EAC_capex_annuity_eur_per_yr,EAC_om_annuity_eur_per_yr,EAC_total_annuity_eur_per_yr,EAC_capex_paper_eur_per_yr,added_AC_users
0,Dynamic rollout + maturity (start_age=5),Base O&M,Trees only (vs current AC),4.536548e+08,97.55,4650345.49,12093350.58,13959075.97,26052426.56,5775851.59,0.00
1,Dynamic rollout + maturity (start_age=5),Base O&M,AC only (vs current AC),1.243785e+09,744.01,1671741.22,0.00,0.00,71427925.36,NaN,199063.06
2,Dynamic rollout + maturity (start_age=5),Base O&M,Both (trees+AC vs current AC),1.697440e+09,835.42,2031851.24,12093350.58,13959075.97,97480351.92,5775851.59,199063.06
3,Dynamic rollout + maturity (start_age=5),Base O&M,"Trees (incremental, on top of AC policy)",4.536548e+08,91.41,4962875.05,12093350.58,13959075.97,26052426.56,5775851.59,0.00
4,Dynamic rollout + maturity (start_age=0),Base O&M,Trees only (vs current AC),3.788200e+08,69.95,5415252.12,12093350.58,9661473.99,21754824.57,5775851.59,0.00
5,Dynamic rollout + maturity (start_age=0),Base O&M,AC only (vs current AC),1.243785e+09,744.01,1671741.22,0.00,0.00,71427925.36,NaN,199063.06
6,Dynamic rollout + maturity (start_age=0),Base O&M,Both (trees+AC vs current AC),1.622605e+09,809.57,2004276.60,12093350.58,9661473.99,93182749.93,5775851.59,199063.06
7,Dynamic rollout + maturity (start_age=0),Base O&M,"Trees (incremental, on top of AC policy)",3.788200e+08,65.57,5777717.84,12093350.58,9661473.99,21754824.57,5775851.59,0.00


- Baseline mortality + percent reductions
  - Read baseline heat-attributable deaths under current AC (no new policy)
  - Convert avoided deaths to % reduction for trees, AC, both, and trees-on-top

In [18]:
# Baseline mortality
from pathlib import Path
import numpy as np
import pandas as pd

def read_baseline_curr_ac(int_dir: Path, slug: str) -> pd.DataFrame:
    """ Reads baseline annual heat-attributable deaths under current AC (no new policy)
    from the NB6 Excel bundle, sheet 'yr_totals_base'.

    Returns a DataFrame indexed by year with a single column: 'baseline_total'.
    """
    xlsx_path = Path(int_dir) / f"annual_heat_deaths_AC_bundle_{slug}.xlsx"
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Missing Excel bundle: {xlsx_path}")

    df = pd.read_excel(xlsx_path, sheet_name="yr_totals_base", index_col=0)
    df.index = df.index.astype(int)
    df = df.sort_index()

    if "overall" in df.columns:
        baseline = df["overall"].astype(float)
    else:
        # sum all numeric columns (typically age groups)
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) == 0:
            raise ValueError(f"'yr_totals_base' has no numeric columns to sum.")
        baseline = df[numeric_cols].sum(axis=1).astype(float)

    return baseline.to_frame("baseline_total")

baseline_df = read_baseline_curr_ac(INT, SLUG)

# align baseline to our horizon
baseline_total = interpolate_to_horizon(baseline_df["baseline_total"], YEARS)

def make_benefit_pct_table(label: str, baseline_total, trees, ac, both, top) -> pd.DataFrame:
    out = pd.DataFrame({
        "year": YEARS,
        "baseline_total": baseline_total,
        "avo_trees": trees,
        "avo_ac": ac,
        "avo_both": both,
        "avo_trees_on_top": top,
    }).set_index("year")
    out["trees_pct"] = 100 * out["avo_trees"] / out["baseline_total"]
    out["ac_pct"] = 100 * out["avo_ac"] / out["baseline_total"]
    out["both_pct"] = 100 * out["avo_both"] / out["baseline_total"]
    # two equivalent ways; this matches old logic
    out["trees_on_top_pct"] = out["both_pct"] - out["ac_pct"]
    # optional: tag timing convention
    out.insert(0, "Benefit_timing", label)
    return out

benefit_pct_dynamic = make_benefit_pct_table(
    "Dynamic rollout + tree maturity",
    baseline_total, trees_dyn, ac_dyn, both_dyn, top_dyn
).round(2)

benefit_pct_static = make_benefit_pct_table(
    "Static (full effect from start)",
    baseline_total, trees_sta, ac_sta, both_sta, top_sta
).round(2)

benefit_pct = pd.concat([benefit_pct_dynamic, benefit_pct_static])
benefit_pct

,Benefit_timing,baseline_total,avo_trees,avo_ac,avo_both,avo_trees_on_top,trees_pct,ac_pct,both_pct,trees_on_top_pct
year,,,,,,,,,,
2030,Dynamic rollout + tree maturity,643.65,0.00,32.19,32.19,0.00,0.00,5.00,5.00,0.00
2031,Dynamic rollout + tree maturity,648.58,0.17,31.78,31.94,0.16,0.03,4.90,4.93,0.02
2032,Dynamic rollout + tree maturity,653.51,0.38,31.38,31.73,0.35,0.06,4.80,4.86,0.05
2033,Dynamic rollout + tree maturity,658.44,0.62,30.98,31.55,0.57,0.09,4.70,4.79,0.09
2034,Dynamic rollout + tree maturity,663.37,0.88,30.57,31.39,0.82,0.13,4.61,4.73,0.12
2035,Dynamic rollout + tree maturity,668.30,1.18,30.17,31.27,1.10,0.18,4.51,4.68,0.16
2036,Dynamic rollout + tree maturity,673.24,1.51,29.76,31.18,1.41,0.22,4.42,4.63,0.21
2037,Dynamic rollout + tree maturity,678.17,1.88,29.36,31.12,1.75,0.28,4.33,4.59,0.26
2038,Dynamic rollout + tree maturity,683.10,2.25,28.96,31.06,2.10,0.33,4.24,4.55,0.31


**On a PV budget**

**Sensitivity**